# Fracture characterisation: HTI and AVAz

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mosesfalowo/ogbonlab-notebooks/blob/main/notebooks/ssm_fracture_avaz_hti.ipynb)


Generated by [OgbonLab](https://ogbonlab.com), Synthetic Seismic Modeling course.

**Assumptions**: Course defaults. Edit the parameters cell and run again; every value is yours to change.

**Workflow**:
1. 8.4 Building a Fractured Model
2. 8.2 Hudson and Linear Slip
3. 8.5 Azimuthal AVO (AVAz)
4. 8.6 Shear-Wave Splitting

In [ ]:
# Ask Ogbon starter: Characterise a fractured reservoir
# Generated by OgbonLab (ogbonlab.com), Synthetic Seismic Modeling course.
import numpy as np
import matplotlib.pyplot as plt

# --- parameters from your interview ---
E_CRACK = 0.05
VS_VP = 1600.0 / 2800.0
NORMAL_MOD = 1.0    # 1.0 = dry or gas fill, ~0.1 = brine fill
TARGET_DEPTH_M = 2500
THETA_DEG = 30.0
INCLUDE_SPLITTING = False
SEED = 3

In [ ]:
def hudson_hti(e, vsvp, normal_mod=1.0):
    g = np.clip(vsvp**2, 0.05, 0.6)
    wN = 4*e / (3*g*(1-g)) * normal_mod
    wT = 16*e / (3*(3 - 2*g))
    dN, dT = wN/(1+wN), wT/(1+wT)
    epsV = -2*g*(1-g)*dN
    deltaV = -2*g*((1-2*g)*dN + dT)
    gammaV = -dT/2
    return epsV, deltaV, gammaV

# Ruger azimuthal AVO: the anisotropic gradient rides on the SYMMETRY
# AXIS (the crack normal), 90 degrees from the fracture strike.
def ruger_rpp(theta_deg, phi_deg, r0, b_iso, b_ani, strike_deg):
    th = np.radians(theta_deg)
    d = np.radians(phi_deg - strike_deg - 90.0)
    return r0 + (b_iso + b_ani * np.cos(d)**2) * np.sin(th)**2

epsV, deltaV, gammaV = hudson_hti(E_CRACK, VS_VP, NORMAL_MOD)
print('HTI params: epsV %.3f  deltaV %.3f  gammaV %.3f' % (epsV, deltaV, gammaV))

STRIKE_TRUE = 50.0
b_ani = 2.5 * deltaV
rng = np.random.default_rng(SEED)
phi = np.linspace(0, 180, 12, endpoint=False)
amp = ruger_rpp(THETA_DEG, phi, 0.10, -0.12, b_ani, STRIKE_TRUE)
amp = amp + rng.normal(0, 0.004, phi.size)   # gauge noise

# least-squares cos(2 phi) ellipse fit recovers the strike
A = np.c_[np.ones_like(phi), np.cos(np.radians(2*phi)), np.sin(np.radians(2*phi))]
coef, *_ = np.linalg.lstsq(A, amp, rcond=None)
strike_rec = np.degrees(0.5 * np.arctan2(coef[2], coef[1])) % 180
print('strike: true %.0f  recovered %.1f' % (STRIKE_TRUE, strike_rec))

if INCLUDE_SPLITTING:
    # fluid-blind check: shear-wave splitting from gammaV
    split_pct = -gammaV * 100
    print('expected shear splitting: %.1f%% slow/fast delay' % split_pct)

phi_f = np.linspace(0, 360, 361)
fit = coef[0] + coef[1]*np.cos(np.radians(2*phi_f)) + coef[2]*np.sin(np.radians(2*phi_f))
ax = plt.subplot(111, projection='polar')
ax.plot(np.radians(phi_f), fit - fit.min() + 0.01)
ax.plot(np.radians(np.r_[phi, phi + 180]),
        np.r_[amp, amp] - fit.min() + 0.01, 'o', ms=4)
ax.set_title('AVAz ellipse: long axis = fracture strike')
plt.tight_layout(); plt.show()

## Where to go next

- The course: https://ogbonlab.com/textbooks/synthetic-seismic-modeling
- More notebooks: https://github.com/mosesfalowo/ogbonlab-notebooks
- Scale up: Devito, SPECFEM3D, Madagascar (course Part 12.3)